In [13]:
import os
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer()
tok.train(
    ["corpus.txt"],
    vocab_size=4000,
    min_frequency=2,
    special_tokens=["<eos>"],
)
os.makedirs("tok", exist_ok=True)   # create the folder if it isn't there
tok.save_model("tok")
print("saved:", os.listdir("tok"))

saved: ['merges.txt', 'vocab.json']


In [14]:
print(os.path.abspath("tok"))

C:\Users\karus\Documents\project\label\tok


In [15]:
eos_id = tok.token_to_id("<eos>")
print("eos id:", eos_id)

sample = "User: who is Karus?\nAssistant: That's Lê Quán Hiếu."
enc = tok.encode(sample)
print(enc.tokens)
print(enc.ids)
print(tok.decode(enc.ids))   # should print the original text back

eos id: 0
['User', ':', 'Ġwho', 'Ġis', 'ĠKarus', '?', 'Ċ', 'Assistant', ':', 'ĠThat', "'s", 'ĠLÃª', 'ĠQuÃ¡n', 'ĠHiáº¿u', '.']
[375, 26, 609, 354, 468, 31, 199, 374, 26, 862, 320, 959, 960, 555, 14]
User: who is Karus?
Assistant: That's Lê Quán Hiếu.


In [16]:
import numpy as np

text = open("corpus.txt", encoding="utf-8").read()
docs = [d.strip() for d in text.split("<eos>") if d.strip()]

encoded = tok.encode_batch(docs)      # encodes many docs at once, much faster
ids = []
for e in encoded:
    ids.extend(e.ids)
    ids.append(eos_id)                # put <eos> back between documents

ids = np.array(ids, dtype=np.uint16)
split = int(len(ids) * 0.9)
ids[:split].tofile("train.bin")
ids[split:].tofile("val.bin")
print(f"{len(ids):,} tokens  |  train {split:,}  |  val {len(ids) - split:,}")

6,474,641 tokens  |  train 5,827,176  |  val 647,465
